In [1]:
%pip install -q sentence-transformers faiss-cpu tqdm


[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

CHUNK_DIR = Path("chunks")
INDEX_DIR = Path("indexes")
INDEX_DIR.mkdir(exist_ok=True)

CHUNK_FILES = {
    "document": CHUNK_DIR / "chunks_document.json",
    "sentence": CHUNK_DIR / "chunks_sentence.json",
    "fixed_128": CHUNK_DIR / "chunks_fixed_128.json",
    "recursive_128": CHUNK_DIR / "chunks_recursive_128.json",
    "recursive_256": CHUNK_DIR / "chunks_recursive_256.json",
    "semantic": CHUNK_DIR / "chunks_semantic.json",
}

# Load embedding model

In [3]:
embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
def build_faiss_index(index_name, chunk_path, batch_size=64):
    print(f"\nBuilding index: {index_name}")
    print(f"Loading chunks from: {chunk_path}")

    with open(chunk_path, "r") as f:
        chunks = json.load(f)

    texts = [c["text"] for c in chunks]

    embeddings = embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
    )

    embeddings = np.asarray(embeddings, dtype="float32")

    dim = embeddings.shape[1]

    # Because embeddings are normalized, inner product is cosine similarity.
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings)

    faiss_path = INDEX_DIR / f"{index_name}.faiss"
    metadata_path = INDEX_DIR / f"{index_name}_chunks.json"

    faiss.write_index(index, str(faiss_path))

    with open(metadata_path, "w") as f:
        json.dump(chunks, f, ensure_ascii=False, indent=2)

    print(f"Saved FAISS index to: {faiss_path}")
    print(f"Saved chunk metadata to: {metadata_path}")
    print(f"Total chunks indexed: {len(chunks)}")
    print(f"Embedding dimension: {dim}")

    return {
        "index_name": index_name,
        "num_chunks": len(chunks),
        "embedding_dim": dim,
        "faiss_path": str(faiss_path),
        "metadata_path": str(metadata_path),
    }

In [5]:
index_summaries = []

for index_name, chunk_path in CHUNK_FILES.items():
    summary = build_faiss_index(index_name, chunk_path)
    index_summaries.append(summary)

index_summary_df = pd.DataFrame(index_summaries)
index_summary_df


Building index: document
Loading chunks from: chunks/chunks_document.json


Batches:   0%|          | 0/81 [00:00<?, ?it/s]

Saved FAISS index to: indexes/document.faiss
Saved chunk metadata to: indexes/document_chunks.json
Total chunks indexed: 5183
Embedding dimension: 384

Building index: sentence
Loading chunks from: chunks/chunks_sentence.json


Batches:   0%|          | 0/718 [00:00<?, ?it/s]

Saved FAISS index to: indexes/sentence.faiss
Saved chunk metadata to: indexes/sentence_chunks.json
Total chunks indexed: 45952
Embedding dimension: 384

Building index: fixed_128
Loading chunks from: chunks/chunks_fixed_128.json


...

Batches:   0%|          | 0/155 [00:00<?, ?it/s]

Saved FAISS index to: indexes/recursive_256.faiss
Saved chunk metadata to: indexes/recursive_256_chunks.json
Total chunks indexed: 9905
Embedding dimension: 384

Building index: semantic
Loading chunks from: chunks/chunks_semantic.json


Batches:   0%|          | 0/166 [00:00<?, ?it/s]

Saved FAISS index to: indexes/semantic.faiss
Saved chunk metadata to: indexes/semantic_chunks.json
Total chunks indexed: 10594
Embedding dimension: 384


,index_name,num_chunks,embedding_dim,faiss_path,metadata_path
0,document,5183,384,indexes/document.faiss,indexes/document_chunks.json
1,sentence,45952,384,indexes/sentence.faiss,indexes/sentence_chunks.json
2,fixed_128,17400,384,indexes/fixed_128.faiss,indexes/fixed_128_chunks.json
3,recursive_128,18638,384,indexes/recursive_128.faiss,indexes/recursive_128_chunks.json
4,recursive_256,9905,384,indexes/recursive_256.faiss,indexes/recursive_256_chunks.json
5,semantic,10594,384,indexes/semantic.faiss,indexes/semantic_chunks.json


In [6]:
index_summary_df.to_csv(INDEX_DIR / "index_summary.csv", index=False)
print(f"Saved index summary to {INDEX_DIR / 'index_summary.csv'}")

Saved index summary to indexes/index_summary.csv


In [7]:
def load_index(index_name):
    index_path = INDEX_DIR / f"{index_name}.faiss"
    chunks_path = INDEX_DIR / f"{index_name}_chunks.json"

    index = faiss.read_index(str(index_path))

    with open(chunks_path, "r") as f:
        chunks = json.load(f)

    return index, chunks


def search_index(query, index, chunks, top_k=5):
    query_text = f"Represent this sentence for searching relevant passages: {query}"

    query_embedding = embedding_model.encode(
        [query_text],
        normalize_embeddings=True,
    )

    query_embedding = np.asarray(query_embedding, dtype="float32")

    scores, indices = index.search(query_embedding, top_k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[int(idx)]

        results.append({
            "score": float(score),
            "chunk_id": chunk["chunk_id"],
            "doc_id": chunk["doc_id"],
            "chunk_type": chunk.get("chunk_type"),
            "text": chunk["text"],
        })

    return results

In [8]:
index, chunks = load_index("document")

query = "Patients with inflammatory bowel disease have increased risk of colorectal cancer."

results = search_index(query, index, chunks, top_k=5)

for r in results:
    print("Score:", r["score"])
    print("Doc ID:", r["doc_id"])
    print("Chunk ID:", r["chunk_id"])
    print("Text:", r["text"][:500])
    print("-" * 100)

Score: 0.7875632643699646
Doc ID: 7613033
Chunk ID: 7613033_doc_0
Text: Aspirin in the chemoprevention of colorectal neoplasia: an overview.. Considerable evidence supports the effectiveness of aspirin for chemoprevention of colorectal cancer (CRC) in addition to its well-established benefits in the prevention of vascular disease. Epidemiologic studies have consistently observed an inverse association between aspirin use and risk of CRC. A recent pooled analysis of a long-term posttrial follow-up of nearly 14,000 patients from four randomized, cardiovascular disease
----------------------------------------------------------------------------------------------------
Score: 0.7740387916564941
Doc ID: 44672703
Chunk ID: 44672703_doc_0
Text: Increased short- and long-term risk of inflammatory bowel disease after salmonella or campylobacter gastroenteritis.. BACKGROUND & AIMS Various commensal enteric and potentially pathogenic bacteria may be involved in the pathogenesis of inflammatory bo

In [9]:
import faiss
import json
from pathlib import Path

INDEX_DIR = Path("indexes")

index_names = [
    "document",
    "sentence",
    "fixed_128",
    "recursive_128",
    "recursive_256",
    "semantic",
]

for name in index_names:
    index_path = INDEX_DIR / f"{name}.faiss"
    chunks_path = INDEX_DIR / f"{name}_chunks.json"

    if not index_path.exists() or not chunks_path.exists():
        print(f"{name}: MISSING")
        continue

    index = faiss.read_index(str(index_path))

    with open(chunks_path) as f:
        chunks = json.load(f)

    print(f"{name}: vectors={index.ntotal}, chunks={len(chunks)}, match={index.ntotal == len(chunks)}")

document: vectors=5183, chunks=5183, match=True
sentence: vectors=45952, chunks=45952, match=True
fixed_128: vectors=17400, chunks=17400, match=True
recursive_128: vectors=18638, chunks=18638, match=True
recursive_256: vectors=9905, chunks=9905, match=True
semantic: vectors=10594, chunks=10594, match=True
